# ⚽ Predict the FIFA World Cup 2026

## 📖 Background

The 2026 FIFA World Cup is one of the biggest sporting events in the world, hosted across the United States, Canada, and Mexico. For the first time, the tournament expands to 48 teams, producing 104 matches across the group stage and knockout rounds.

Using machine learning, historical statistics, and soccer domain knowledge, predict match scores, corners, and cards for every fixture. You must submit all your predictions before a single ball is kicked.

The scoring system rewards precision: an exact scoreline earns maximum points, while close predictions still earn partial credit. Later rounds carry score multipliers, so a strong model that holds up in the knockout stages can leapfrog the competition. The challenge is designed to be difficult enough that no one can achieve a perfect score—even with AI assistance—but accessible enough that any data enthusiast can participate and score points.

## 💾 The data

You have access to the following files:

#### `data/group_fixtures.csv` — all 72 group stage matches
| Variable | Description |
|---|---|
| `match_id` | Unique match identifier |
| `group` | Group letter (A–L) |
| `home_team` | Home team name |
| `away_team` | Away team name |
| `date` | Match date (UTC) |
| `venue` | Stadium and city |

#### `data/knockout_slots.csv` — all 32 knockout round slots
| Variable | Description |
|---|---|
| `match_id` | Unique match identifier |
| `round` | Round name (e.g. `Quarter-final`) |
| `multiplier` | Score multiplier for this round |
| `slot_home` | Description of the home team slot (e.g. `Winner Group A`) |
| `slot_away` | Description of the away team slot |

| Variable | Description |
|---|---|

You may also bring in any external data—FIFA rankings, historical match results, player statistics—to build your predictions.

In [21]:
import pandas as pd

group_fixtures = pd.read_csv('data/group_fixtures.csv')
group_fixtures.head()

,match_id,group,home_team,away_team,date_utc,venue
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City"
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara"
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto"
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles"
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver"


In [22]:
knockout_slots = pd.read_csv('data/knockout_slots.csv')
knockout_slots

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B
1,74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F
2,75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F)
3,76,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C
4,77,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I
5,78,Round of 32,1,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Winner Group I,Best 3rd (Groups C/D/F/G/H)
6,79,Round of 32,1,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I)
7,80,Round of 32,1,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K)
8,81,Round of 32,1,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group G,Best 3rd (Groups A/E/H/I/J)
9,82,Round of 32,1,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group D,Best 3rd (Groups B/E/F/I/J)


## 💪 Competition challenge

The 2026 World Cup has two phases:

- **Group stage** (matches 1–72): The 48 teams are split into 12 groups of 4. Every team plays the other 3 teams in their group once. The best teams from each group advance to the next phase.
- **Knockout stage** (matches 73–104): Single-elimination rounds — Round of 32, Round of 16, Quarter-finals, Semi-finals, and the Final. Lose once and you're out. Crucially, the two teams playing in each knockout match are not known in advance: they depend on who qualified from the group stage.

Submit predictions for **every match** in both phases. For each match you need to predict:

1. **Score** — the exact final scoreline (e.g. `2-1` means the home team scores 2, the away team scores 1). For knockout matches, the score is the result after 90 minutes and extra time — the penalty shootout is not included.
2. **Corners** — the number of corner kicks awarded in the match
3. **Yellow cards** — the number of yellow cards shown in the match
4. **Red cards** — the number of red cards shown in the match

For **group stage** matches, also predict:
- **Winning team** — which team wins the individual match (use `home`, `away`, or `draw`)

For **knockout round** matches, also predict:
- **Matchup** — which two teams you predict will be playing in that slot. Because the bracket is determined by group stage results, you need to predict which teams advance far enough to meet in each round.
- **Match winner** — which team wins the match (use `home` or `away`)
- **Penalties** — whether the match goes to a penalty shootout (`True` or `False`)

### Scoring system

| Category | Condition | Points |
|---|---|---|
| Score | Exact scoreline | 25 |
| Score | Correct goal difference, wrong score | 10 |
| Score | Correct total goals, wrong score | 10 |
| Corners | Exact number | 10 |
| Corners | Off by 2 | 5 |
| Yellow cards | Exact number | 10 |
| Yellow cards | Off by 1 | 5 |
| Red cards | Exact number | 5 |
| Winning team *(group stage only)* | Correct | 40 |
| Matchup *(knockout only)* | Both teams correct | 20 |
| Matchup *(knockout only)* | One team correct | 10 |
| Match winner *(knockout only)* | Correct | 20 |
| Penalties *(knockout only)* | Correct | 5 |

All points for a match are multiplied by the round factor:

| Round | Multiplier |
|---|---|
| Group stage | ×1 |
| Round of 32 | ×1 |
| Round of 16 | ×2 |
| Quarter-final | ×4 |
| Semi-final | ×8 |
| Third-place playoff | ×8 |
| Final | ×16 |

## 🗓️ Group stage predictions

Fill in your predictions for all 72 group stage matches below.

In [23]:
group_predictions = group_fixtures.copy()

# Fill in your predictions for each match
# Example (match 1 — Mexico vs South Africa): predicted_home_goals=2, predicted_away_goals=1, corners=9, yellow_cards=3, red_cards=0, winning_team='home'
group_predictions['predicted_home_goals'] = None   # e.g. 2
group_predictions['predicted_away_goals'] = None   # e.g. 1
group_predictions['corners']              = None   # e.g. 9
group_predictions['yellow_cards']         = None   # e.g. 3
group_predictions['red_cards']            = None   # e.g. 0
group_predictions['winning_team']         = None   # "home", "away", or "draw"

group_predictions

,match_id,group,home_team,away_team,date_utc,venue,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,winning_team
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City",None,None,None,None,None,None
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara",None,None,None,None,None,None
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto",None,None,None,None,None,None
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles",None,None,None,None,None,None
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver",None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...
67,68,L,Croatia,Ghana,2026-06-27T21:00:00Z,"Lincoln Financial Field, Philadelphia",None,None,None,None,None,None
68,69,K,Colombia,Portugal,2026-06-27T23:30:00Z,"Hard Rock Stadium, Miami",None,None,None,None,None,None
69,70,K,FIFA Playoff 1,Uzbekistan,2026-06-27T23:30:00Z,"Mercedes-Benz Stadium, Atlanta",None,None,None,None,None,None
70,71,J,Algeria,Austria,2026-06-28T02:00:00Z,"GEHA Field at Arrowhead Stadium, Kansas City",None,None,None,None,None,None


## 🏆 Knockout stage predictions

For knockout matches you also predict **which teams are playing**. Fill in the team names based on your group stage predictions, then add your match predictions.

In [24]:
knockout_predictions = knockout_slots.copy()

# Fill in your predictions for each knockout match
# Example (match 73 — Round of 32): predicted_home_team='Brazil', predicted_away_team='France', predicted_home_goals=1, predicted_away_goals=0, corners=8, yellow_cards=2, red_cards=0, match_winner='home', penalties=False
knockout_predictions['predicted_home_team']  = None   # e.g. "Brazil"
knockout_predictions['predicted_away_team']  = None   # e.g. "France"
knockout_predictions['predicted_home_goals'] = None   # e.g. 1
knockout_predictions['predicted_away_goals'] = None   # e.g. 0
knockout_predictions['corners']              = None   # e.g. 8
knockout_predictions['yellow_cards']         = None   # e.g. 2
knockout_predictions['red_cards']            = None   # e.g. 0
knockout_predictions['match_winner']         = None   # "home" or "away"
knockout_predictions['penalties']            = None   # True or False

knockout_predictions

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away,predicted_home_team,predicted_away_team,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,match_winner,penalties
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B,None,None,None,None,None,None,None,None,None
1,74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F,None,None,None,None,None,None,None,None,None
2,75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F),None,None,None,None,None,None,None,None,None
3,76,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C,None,None,None,None,None,None,None,None,None
4,77,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I,None,None,None,None,None,None,None,None,None
5,78,Round of 32,1,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Winner Group I,Best 3rd (Groups C/D/F/G/H),None,None,None,None,None,None,None,None,None
6,79,Round of 32,1,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I),None,None,None,None,None,None,None,None,None
7,80,Round of 32,1,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K),None,None,None,None,None,None,None,None,None
8,81,Round of 32,1,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group G,Best 3rd (Groups A/E/H/I/J),None,None,None,None,None,None,None,None,None
9,82,Round of 32,1,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group D,Best 3rd (Groups B/E/F/I/J),None,None,None,None,None,None,None,None,None


## ✅ Checklist before publishing into the competition

- Rename your workspace to make it descriptive of your work. N.B. you should leave the notebook name as `notebook.ipynb`.
- Remove redundant cells like the judging criteria, so the workbook is focused on your predictions.
- Make sure all prediction cells are filled in—`None` values will score 0 points.
- Check that all cells run without error.
- Make sure your workbook is published before **June 10, 2026 at 09:00 UTC**.

## ⏳ Time is ticking. Good luck!

# 2026 FIFA World Cup predictions

Starting from here there is all the code I used for developing a simulation of the 2026 FIFA World Cup.

### Libraries

In [25]:
import pandas as pd
import pandas as pd
import numpy as np
import requests
from io import StringIO
from sklearn.linear_model import PoissonRegressor
import statsmodels.api as sm
from xgboost import XGBRegressor

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

from sklearn.utils.class_weight import compute_sample_weight
#from skopt import BayesSearchCV, space

## General approach description

My workflow started from gathering training data, specifically I considered data from the Kaggle-Internationa football results dataset (stored in the results.csv file, also available online at https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017?select=results.csv), from the Elo ranking of national teams (https://www.eloratings.net/) and from the FIFA ranking (https://inside.fifa.com/fifa-world-ranking)
The feature I included are:

    1) Position in the FIFA ranking.
    2) Elo rating.
    3) Last results and feuares obtained from these results. 


The model was trained on the data starting from 2008 and the model developed was used to simulate the last four World Cups for testing (data regarding these tournaments were not included in the training data).

The models are a XGBoost classifier for the winning team prediction and zero inflated Poisson models for the final result. Due to lack of time corners, red and yellow cards are computed as the average number of such events in the World Cup tournaments.

## Collecting the data

The following code chuncks contain the code needed for obtaining all the data.

In [26]:
results = pd.read_csv('data/results.csv')

results['date'] = pd.to_datetime(results['date'], format='%Y-%m-%d')

world_cup = results[results['tournament'] == 'FIFA World Cup']

In [27]:
# let's get the data for the 2010 World Cup

tournament_2010 = world_cup[(world_cup['date'] >= '2010-01-01' ) & (world_cup['date'] <= '2011-01-01')]

teams_2010 = tournament_2010['home_team'].unique()

teams_mask_2010 = (results.home_team.isin(teams_2010)) | (results.away_team.isin(teams_2010))

last_results_2010 = results[teams_mask_2010 & ((results.date >= pd.to_datetime('2008-01-01')) & (results.date <= pd.to_datetime('2010-07-11')))]

# we want to use the non world cup games as indicators of the the status of teams to build the features 
# we need and then use these to train the model to predict the results.

non_wc_2010 = last_results_2010[last_results_2010['tournament'] != 'FIFA World Cup']
wc_2010 = last_results_2010[last_results_2010['tournament'] == 'FIFA World Cup']

# Now that the games are split we can build the feuatures that can be useful, like:
# - average goals scored
# - average goals conceded
# - win rate
# - draw rate

# In order to obtain all these features we have to duplicate the df, one for the home team and one for the away team. 

home_2010 = non_wc_2010[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament']].copy()
home_2010.columns = ['date', 'team', 'opponent', 'goals_for', 'goals_against', 'tournament']


away_2010 = non_wc_2010[['date', 'away_team', 'home_team', 'away_score', 'home_score', 'tournament']].copy()
away_2010.columns = ['date', 'team', 'opponent', 'goals_for', 'goals_against', 'tournament']


long_df_2010 = pd.concat([home_2010, away_2010], axis=0)

# we have now a full dataframe with two entries for each game that will be used to compute all we need.
# at first let's sort the row for date and team
long_df_2010 = long_df_2010.sort_values(by=['team', 'date']).reset_index(drop=True)

# and now let's exploit this new structure to compute the features


groups = long_df_2010.groupby('team')
mask = [long_df_2010.goals_for > long_df_2010.goals_against, 
        long_df_2010.goals_for == long_df_2010.goals_against,
        long_df_2010.goals_for < long_df_2010.goals_against]

long_df_2010['result'] = np.select(mask, ['W', 'D', 'L'], default='N/A')

long_df_2010['match_played'] = groups.cumcount()
long_df_2010['average_goal'] = groups['goals_for'].transform(lambda x: x.cumsum().shift(1))/groups['match_played'].fillna(0)
long_df_2010['average_goal_conceded'] = groups['goals_against'].transform(lambda x: x.cumsum().shift(1))/groups['match_played'].fillna(0)


long_df_2010['is_win'] = np.where(long_df_2010['result'] == 'W', 1, 0)
long_df_2010['is_draw'] = np.where(long_df_2010['result'] == 'D', 1, 0)

long_df_2010['win_rate'] = (groups['is_win'].transform(lambda x: x.cumsum().shift(1)) / long_df_2010['match_played']).fillna(0)

long_df_2010['draw_rate'] = (groups['is_draw'].transform(lambda df: df.cumsum().shift(1)) / long_df_2010['match_played']).fillna(0)




C:\Users\emanu\AppData\Local\Temp\ipykernel_8300\2869856934.py:50: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  long_df_2010['average_goal'] = groups['goals_for'].transform(lambda x: x.cumsum().shift(1))/groups['match_played'].fillna(0)
C:\Users\emanu\AppData\Local\Temp\ipykernel_8300\2869856934.py:51: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  long_df_2010['average_goal_conceded'] = groups['goals_against'].transform(lambda x: x.cumsum().shift(1))/groups['match_played'].fillna(0)


In [28]:
# let's get the data for the 2014 World Cup

tournament_2014 = world_cup[(world_cup['date'] >= '2014-01-01' ) & (world_cup['date'] <= '2015-01-01')]

teams_2014 = tournament_2014['home_team'].unique()

teams_mask_2014 = (results.home_team.isin(teams_2014)) | (results.away_team.isin(teams_2014))

last_results_2014 = results[teams_mask_2014 & ((results.date >= pd.to_datetime('2012-01-01')) & (results.date <= pd.to_datetime('2014-07-13')))]

# we want to use the non world cup games as indicators of the the status of teams to build the features 
# we need and then use these to train the model to predict the results.

non_wc_2014 = last_results_2014[last_results_2014['tournament'] != 'FIFA World Cup']
wc_2014 = last_results_2014[last_results_2014['tournament'] == 'FIFA World Cup']

# Now that the games are split we can build the feuatures that can be useful, like:
# - average goals scored
# - average goals conceded
# - win rate
# - draw rate

# In order to obtain all these features we have to duplicate the df, one for the home team and one for the away team. 

home_2014 = non_wc_2014[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament']].copy()
home_2014.columns = ['date', 'team', 'opponent', 'goals_for', 'goals_against', 'tournament']


away_2014 = non_wc_2014[['date', 'away_team', 'home_team', 'away_score', 'home_score', 'tournament']].copy()
away_2014.columns = ['date', 'team', 'opponent', 'goals_for', 'goals_against', 'tournament']


long_df_2014 = pd.concat([home_2014, away_2014], axis=0)

# we have now a full dataframe with two entries for each game that will be used to compute all we need.
# at first let's sort the row for date and team
long_df_2014 = long_df_2014.sort_values(by=['team', 'date']).reset_index(drop=True)

# and now let's exploit this new structure to compute the features


groups = long_df_2014.groupby('team')
mask = [long_df_2014.goals_for > long_df_2014.goals_against, 
        long_df_2014.goals_for == long_df_2014.goals_against,
        long_df_2014.goals_for < long_df_2014.goals_against]

long_df_2014['result'] = np.select(mask, ['W', 'D', 'L'], default='N/A')

long_df_2014['match_played'] = groups.cumcount()
long_df_2014['average_goal'] = groups['goals_for'].transform(lambda x: x.cumsum().shift(1))/groups['match_played'].fillna(0)
long_df_2014['average_goal_conceded'] = groups['goals_against'].transform(lambda x: x.cumsum().shift(1))/groups['match_played'].fillna(0)


long_df_2014['is_win'] = np.where(long_df_2014['result'] == 'W', 1, 0)
long_df_2014['is_draw'] = np.where(long_df_2014['result'] == 'D', 1, 0)

long_df_2014['win_rate'] = (groups['is_win'].transform(lambda x: x.cumsum().shift(1)) / long_df_2014['match_played']).fillna(0)

long_df_2014['draw_rate'] = (groups['is_draw'].transform(lambda x: x.cumsum().shift(1)) / long_df_2014['match_played']).fillna(0)

C:\Users\emanu\AppData\Local\Temp\ipykernel_8300\1433184568.py:50: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  long_df_2014['average_goal'] = groups['goals_for'].transform(lambda x: x.cumsum().shift(1))/groups['match_played'].fillna(0)
C:\Users\emanu\AppData\Local\Temp\ipykernel_8300\1433184568.py:51: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  long_df_2014['average_goal_conceded'] = groups['goals_against'].transform(lambda x: x.cumsum().shift(1))/groups['match_played'].fillna(0)


In [29]:
# let's get the data for the 2018 World Cup

tournament_2018 = world_cup[(world_cup['date'] >= '2018-01-01' ) & (world_cup['date'] <= '2019-01-01')]

teams_2018 = tournament_2018['home_team'].unique()

teams_mask_2018 = (results.home_team.isin(teams_2018)) | (results.away_team.isin(teams_2018))

last_results_2018 = results[teams_mask_2018 & ((results.date >= pd.to_datetime('2016-01-01')) & (results.date <= pd.to_datetime('2018-07-16')))]

# we want to use the non world cup games as indicators of the the status of teams to build the features 
# we need and then use these to train the model to predict the results.

non_wc_2018 = last_results_2018[last_results_2018['tournament'] != 'FIFA World Cup']
wc_2018 = last_results_2018[last_results_2018['tournament'] == 'FIFA World Cup']

# Now that the games are split we can build the feuatures that can be useful, like:
# - average goals scored
# - average goals conceded
# - win rate
# - draw rate

# In order to obtain all these features we have to duplicate the df, one for the home team and one for the away team. 

home_2018 = non_wc_2018[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament']].copy()
home_2018.columns = ['date', 'team', 'opponent', 'goals_for', 'goals_against', 'tournament']


away_2018 = non_wc_2018[['date', 'away_team', 'home_team', 'away_score', 'home_score', 'tournament']].copy()
away_2018.columns = ['date', 'team', 'opponent', 'goals_for', 'goals_against', 'tournament']


long_df_2018 = pd.concat([home_2018, away_2018], axis=0)

# we have now a full dataframe with two entries for each game that will be used to compute all we need.
# at first let's sort the row for date and team
long_df_2018 = long_df_2018.sort_values(by=['team', 'date']).reset_index(drop=True)

# and now let's exploit this new structure to compute the features


groups = long_df_2018.groupby('team')
mask = [long_df_2018.goals_for > long_df_2018.goals_against, 
        long_df_2018.goals_for == long_df_2018.goals_against,
        long_df_2018.goals_for < long_df_2018.goals_against]

long_df_2018['result'] = np.select(mask, ['W', 'D', 'L'], default='N/A')

long_df_2018['match_played'] = groups.cumcount()
long_df_2018['average_goal'] = groups['goals_for'].transform(lambda x: x.cumsum().shift(1))/groups['match_played'].fillna(0)
long_df_2018['average_goal_conceded'] = groups['goals_against'].transform(lambda x: x.cumsum().shift(1))/groups['match_played'].fillna(0)


long_df_2018['is_win'] = np.where(long_df_2018['result'] == 'W', 1, 0)
long_df_2018['is_draw'] = np.where(long_df_2018['result'] == 'D', 1, 0)

long_df_2018['win_rate'] = (groups['is_win'].transform(lambda x: x.cumsum().shift(1)) / long_df_2018['match_played']).fillna(0)

long_df_2018['draw_rate'] = (groups['is_draw'].transform(lambda x: x.cumsum().shift(1)) / long_df_2018['match_played']).fillna(0)

C:\Users\emanu\AppData\Local\Temp\ipykernel_8300\2205560816.py:50: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  long_df_2018['average_goal'] = groups['goals_for'].transform(lambda x: x.cumsum().shift(1))/groups['match_played'].fillna(0)
C:\Users\emanu\AppData\Local\Temp\ipykernel_8300\2205560816.py:51: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  long_df_2018['average_goal_conceded'] = groups['goals_against'].transform(lambda x: x.cumsum().shift(1))/groups['match_played'].fillna(0)


In [30]:
# let's get the data for the 2022 World Cup

tournament_2022 = world_cup[(world_cup['date'] >= '2022-01-01' ) & (world_cup['date'] <= '2023-01-01')]

teams_2022 = tournament_2022['home_team'].unique()

teams_mask_2022 = (results.home_team.isin(teams_2022)) | (results.away_team.isin(teams_2022))

last_results_2022 = results[teams_mask_2022 & ((results.date >= pd.to_datetime('2020-01-01')) & (results.date <= pd.to_datetime('2022-12-18')))]

# we want to use the non world cup games as indicators of the the status of teams to build the features 
# we need and then use these to train the model to predict the results.

non_wc_2022 = last_results_2022[last_results_2022['tournament'] != 'FIFA World Cup']
wc_2022 = last_results_2022[last_results_2022['tournament'] == 'FIFA World Cup']

# Now that the games are split we can build the feuatures that can be useful, like:
# - average goals scored
# - average goals conceded
# - win rate
# - draw rate

# In order to obtain all these features we have to duplicate the df, one for the home team and one for the away team. 

home_2022 = non_wc_2022[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament']].copy()
home_2022.columns = ['date', 'team', 'opponent', 'goals_for', 'goals_against', 'tournament']


away_2022 = non_wc_2022[['date', 'away_team', 'home_team', 'away_score', 'home_score', 'tournament']].copy()
away_2022.columns = ['date', 'team', 'opponent', 'goals_for', 'goals_against', 'tournament']


long_df_2022 = pd.concat([home_2022, away_2022], axis=0)

# we have now a full dataframe with two entries for each game that will be used to compute all we need.
# at first let's sort the row for date and team
long_df_2022 = long_df_2022.sort_values(by=['team', 'date']).reset_index(drop=True)

# and now let's exploit this new structure to compute the features


groups = long_df_2022.groupby('team')
mask = [long_df_2022.goals_for > long_df_2022.goals_against, 
        long_df_2022.goals_for == long_df_2022.goals_against,
        long_df_2022.goals_for < long_df_2022.goals_against]

long_df_2022['result'] = np.select(mask, ['W', 'D', 'L'], default='N/A')

long_df_2022['match_played'] = groups.cumcount()
long_df_2022['average_goal'] = groups['goals_for'].transform(lambda x: x.cumsum().shift(1))/groups['match_played'].fillna(0)
long_df_2022['average_goal_conceded'] = groups['goals_against'].transform(lambda x: x.cumsum().shift(1))/groups['match_played'].fillna(0)


long_df_2022['is_win'] = np.where(long_df_2022['result'] == 'W', 1, 0)
long_df_2022['is_draw'] = np.where(long_df_2022['result'] == 'D', 1, 0)

long_df_2022['win_rate'] = (groups['is_win'].transform(lambda x: x.cumsum().shift(1)) / long_df_2022['match_played']).fillna(0)

long_df_2022['draw_rate'] = (groups['is_draw'].transform(lambda x: x.cumsum().shift(1)) / long_df_2022['match_played']).fillna(0)



C:\Users\emanu\AppData\Local\Temp\ipykernel_8300\3414774347.py:50: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  long_df_2022['average_goal'] = groups['goals_for'].transform(lambda x: x.cumsum().shift(1))/groups['match_played'].fillna(0)
C:\Users\emanu\AppData\Local\Temp\ipykernel_8300\3414774347.py:51: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  long_df_2022['average_goal_conceded'] = groups['goals_against'].transform(lambda x: x.cumsum().shift(1))/groups['match_played'].fillna(0)


In [31]:
# ELO rating 

years = [2008, 2009, 2010, 2013, 2014, 2017, 2018, 2021, 2022]

columns = [
    'rank_change', 'rank', 'country_code', 'elo',
    'rank_prev_year', 'elo_prev_year',
    'rank_5y', 'elo_5y', 'rank_10y', 'elo_10y',
    'rank_change_1m', 'elo_change_1m',
    'rank_change_3m', 'elo_change_3m',
    'rank_change_6m', 'elo_change_6m',
    'rank_change_1y', 'elo_change_1y',
    'rank_change_2y', 'elo_change_2y',
    'rank_change_5y', 'elo_change_5y',
    'matches', 'wins', 'draws', 'losses',
    'goals_for', 'goals_against', 'goal_diff',
    'home_goals_for', 'home_goals_against',
    'rank_change_1m_alt', 'elo_change_1m_alt'
]

dfs = []
for year in years:
    url = f"https://www.eloratings.net/{year}.tsv"
    response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
    response.encoding = 'utf-8'
    df = pd.read_csv(StringIO(response.text), sep='\t', header=None)
    df.columns = columns
    df['year'] = year
    dfs.append(df)

elo_all = pd.concat(dfs, ignore_index=True)

elo_features = elo_all[['country_code', 'elo', 'elo_change_1y', 'year']]


In [32]:
# id of pre WC matches
date_ids_by_year = {
    2008: {
        "Jan": "id8415", "Feb": "id8443", "Mar": "id8471", "Apr": "id8499",
        "May": "id8527", "Jun": "id8555", "Jul": "id8583", "Aug": "id8618",
        "Sep": "id8646", "Oct": "id8681", "Nov": "id8716", "Dec": "id8751"
    },
    2009: {
        "Jan": "id8779", "Feb": "id8807", "Mar": "id8835", "Apr": "id8863",
        "May": "id8891", "Jun": "id8919", "Jul": "id8947", "Aug": "id8982",
        "Sep": "id9010", "Oct": "id9054", "Nov": "id9089", "Dec": "id9115"
    },
    2010: {
        "Feb": "id9164", "Mar1": "id9192", "Mar2": "id9220", "Apr": "id9248",
        "May": "id9276", "Jul1": "id9325", "Aug": "id9353", "Sep": "id9388",
        "Oct": "id9423", "Nov": "id9451", "Dec": "id9479"
    },
    2013: {
        "Jan": "id10243", "Feb": "id10271", "Mar": "id10299", "Apr": "id10327",
        "May": "id10355", "Jun": "id10383", "Jul": "id10411", "Aug": "id10446",
        "Sep": "id10481", "Oct": "id10516", "Nov": "id10558", "Dec": "id10579"
    },
    2014: {
        "Jan": "id10607", "Feb": "id10635", "Mar": "id10663", "Apr": "id10691",
        "May": "id10719", "Jun1": "id10747", "Jul": "id10789", "Aug": "id10817",
        "Sep": "id10852", "Oct": "id10887", "Nov": "id10922", "Dec": "id10943"
    },
    2017: {
        "Jan": "id11699", "Feb": "id11727", "Mar": "id11755", "Apr": "id11783",
        "May": "id11811", "Jun": "id11839", "Jul": "id11874", "Aug": "id11909",
        "Sep": "id11944", "Oct": "id11976", "Nov": "id12014", "Dec": "id12042"
    },
    2018: {
        "Jan": "id12070", "Feb": "id12098", "Mar": "id12126", "Apr": "id12154",
        "May": "id12189", "Jun1": "id12210", "Jun2": "id12252", "Aug": "id12280",
        "Sep": "id12315", "Oct": "id12350", "Nov": "id12385", "Dec": "id12406"
    },
    2021: {
        "Feb": "id13197", "Apr": "id13245", "May": "id13295", "Aug": "id13372",
        "Sep": "id13407", "Oct": "id13442", "Nov": "id13471", "Dec": "id13505"
    },
    2022: {
        "Feb": "id13554", "Mar": "id13603", "Jun": "id13687", "Aug": "id13750",
        "Oct": "id13792", "Dec": "id13869"
    }
}

id_to_date = {
    "id8415": "2008-01-16", "id8443": "2008-02-13", "id8471": "2008-03-12",
    "id8499": "2008-04-09", "id8527": "2008-05-07", "id8555": "2008-06-04",
    "id8583": "2008-07-02", "id8618": "2008-08-06", "id8646": "2008-09-03",
    "id8681": "2008-10-08", "id8716": "2008-11-12", "id8751": "2008-12-17",
    "id8779": "2009-01-14", "id8807": "2009-02-11", "id8835": "2009-03-11",
    "id8863": "2009-04-08", "id8891": "2009-05-06", "id8919": "2009-06-03",
    "id8947": "2009-07-01", "id8982": "2009-08-05", "id9010": "2009-09-02",
    "id9054": "2009-10-16", "id9089": "2009-11-20", "id9115": "2009-12-16",
    "id9164": "2010-02-03", "id9192": "2010-03-03", "id9220": "2010-03-31",
    "id9248": "2010-04-28", "id9276": "2010-05-26", "id9325": "2010-07-14",
    "id9353": "2010-08-11", "id9388": "2010-09-15", "id9423": "2010-10-20",
    "id9451": "2010-11-17", "id9479": "2010-12-15",
    "id10243": "2013-01-17", "id10271": "2013-02-14", "id10299": "2013-03-14",
    "id10327": "2013-04-11", "id10355": "2013-05-09", "id10383": "2013-06-06",
    "id10411": "2013-07-04", "id10446": "2013-08-08", "id10481": "2013-09-12",
    "id10516": "2013-10-17", "id10558": "2013-11-28", "id10579": "2013-12-19",
    "id10607": "2014-01-16", "id10635": "2014-02-13", "id10663": "2014-03-13",
    "id10691": "2014-04-10", "id10719": "2014-05-08", "id10747": "2014-06-05",
    "id10789": "2014-07-17", "id10817": "2014-08-14", "id10852": "2014-09-18",
    "id10887": "2014-10-23", "id10922": "2014-11-27", "id10943": "2014-12-18",
    "id11699": "2017-01-12", "id11727": "2017-02-09", "id11755": "2017-03-09",
    "id11783": "2017-04-06", "id11811": "2017-05-04", "id11839": "2017-06-01",
    "id11874": "2017-07-06", "id11909": "2017-08-10", "id11944": "2017-09-14",
    "id11976": "2017-10-16", "id12014": "2017-11-23", "id12042": "2017-12-21",
    "id12070": "2018-01-18", "id12098": "2018-02-15", "id12126": "2018-03-15",
    "id12154": "2018-04-12", "id12189": "2018-05-17", "id12210": "2018-06-07",
    "id12252": "2018-07-01", "id12280": "2018-08-16", "id12315": "2018-09-20",
    "id12350": "2018-10-25", "id12385": "2018-11-29", "id12406": "2018-12-20",
    "id13197": "2021-02-18", "id13245": "2021-04-07", "id13295": "2021-05-27",
    "id13372": "2021-08-12", "id13407": "2021-09-16", "id13442": "2021-10-21",
    "id13471": "2021-11-19", "id13505": "2021-12-23",
    "id13554": "2022-02-10", "id13603": "2022-03-31", "id13687": "2022-06-23",
    "id13750": "2022-08-25", "id13792": "2022-10-06", "id13869": "2022-12-22"
}

def get_fifa_ranking(date_id, date):
    url = f"https://inside.fifa.com/api/ranking-overview?lang=en&dateId={date_id}"
    response = requests.get(url, headers={
        'User-Agent': 'Mozilla/5.0',
        'Accept': 'application/json',
        'Referer': 'https://inside.fifa.com/fifa-world-ranking/men'
    })
    data = response.json()
    records = []
    for item in data['rankings']:
        r = item['rankingItem']
        records.append({
            'date':         pd.to_datetime(date),
            'country_code': r['countryCode'],
            'team_name':    r['name'],
            'fifa_rank':    r['rank'],
            'fifa_points':  r['totalPoints'],
        })
    return pd.DataFrame(records)

all_dfs = []
for date_id, date in id_to_date.items():
    df = get_fifa_ranking(date_id, date)
    all_dfs.append(df)
    

fifa_rankings_all = pd.concat(all_dfs, ignore_index=True)
fifa_rankings_all = fifa_rankings_all.sort_values(['team_name', 'date']).reset_index(drop=True)


In [33]:
fifa_rankings_all['year'] = fifa_rankings_all['date'].dt.year

groups = fifa_rankings_all.groupby(['team_name', 'year'])

fifa_rankings_all['average_position'] = groups['fifa_rank'].transform('mean')
fifa_rankings_all['average_points'] = groups['fifa_points'].transform('mean')

fifa_rankings = fifa_rankings_all[['year', 'country_code', 'team_name', 'average_position', 'average_points']].copy()

fifa_rankings = fifa_rankings.drop_duplicates(subset=['year', 'team_name'], keep='last')


Now all the data are collected, it's time to merge everything

In [34]:

fifa_to_elo = {
    'AFG': 'AF', 'ALB': 'AL', 'ALG': 'DZ', 'ASA': 'AS', 'AND': 'AD',
    'ANG': 'AO', 'AIA': 'AI', 'ATG': 'AG', 'ARG': 'AR', 'ARM': 'AM',
    'ARU': 'AW', 'AUS': 'AU', 'AUT': 'AT', 'AZE': 'AZ', 'BAH': 'BS',
    'BHR': 'BH', 'BAN': 'BD', 'BRB': 'BB', 'BLR': 'BY', 'BEL': 'BE',
    'BLZ': 'BZ', 'BEN': 'BJ', 'BER': 'BM', 'BHU': 'BT', 'BOL': 'BO',
    'BIH': 'BA', 'BOT': 'BW', 'BRA': 'BR', 'VGB': 'VG', 'BRU': 'BN',
    'BUL': 'BG', 'BFA': 'BF', 'BDI': 'BI', 'CPV': 'CV', 'CAM': 'KH',
    'CMR': 'CM', 'CAN': 'CA', 'CAY': 'KY', 'CTA': 'CF', 'CHA': 'TD',
    'CHI': 'CL', 'CHN': 'CN', 'TPE': 'TW', 'COL': 'CO', 'COM': 'KM',
    'CGO': 'CG', 'COD': 'CD', 'COK': 'CK', 'CRC': 'CR', 'CRO': 'HR',
    'CUB': 'CU', 'CUW': 'CW', 'CYP': 'CY', 'CZE': 'CZ', 'CIV': 'CI',
    'DEN': 'DK', 'DJI': 'DJ', 'DMA': 'DM', 'DOM': 'DO', 'ECU': 'EC',
    'EGY': 'EG', 'SLV': 'SV', 'ENG': 'EN', 'EQG': 'GQ', 'ERI': 'ER',
    'EST': 'EE', 'SWZ': 'SZ', 'ETH': 'ET', 'MKD': 'MK', 'FRO': 'FO',
    'FIJ': 'FJ', 'FIN': 'FI', 'FRA': 'FR', 'GAB': 'GA', 'GAM': 'GM',
    'GEO': 'GE', 'GER': 'DE', 'GHA': 'GH', 'GIB': 'GI', 'GRE': 'GR',
    'GRN': 'GD', 'GUM': 'GU', 'GUA': 'GT', 'GUI': 'GN', 'GNB': 'GW',
    'GUY': 'GY', 'HAI': 'HT', 'HON': 'HN', 'HKG': 'HK', 'HUN': 'HU',
    'IRN': 'IR', 'ISL': 'IS', 'IND': 'IN', 'IDN': 'ID', 'IRQ': 'IQ',
    'ISR': 'IL', 'ITA': 'IT', 'JAM': 'JM', 'JPN': 'JP', 'JOR': 'JO',
    'KAZ': 'KZ', 'KEN': 'KE', 'PRK': 'KP', 'KOR': 'KR', 'KOS': 'KO',
    'KUW': 'KW', 'KGZ': 'KG', 'LAO': 'LA', 'LVA': 'LV', 'LBN': 'LB',
    'LIB': 'LY', 'LES': 'LS', 'LBR': 'LR', 'LBY': 'LY', 'LIE': 'LI',
    'LTU': 'LT', 'LUX': 'LU', 'MAC': 'MO', 'MAD': 'MG', 'MWI': 'MW',
    'MAS': 'MY', 'MDV': 'MV', 'MLI': 'ML', 'MLT': 'MT', 'MTN': 'MR',
    'MRI': 'MU', 'MEX': 'MX', 'MDA': 'MD', 'MNG': 'MN', 'MNE': 'ME',
    'MSR': 'MS', 'MAR': 'MA', 'MOZ': 'MZ', 'MYA': 'MM', 'NAM': 'NM',
    'NEP': 'NP', 'NED': 'NL', 'ANT': 'AN', 'NCL': 'NC', 'NZL': 'NZ',
    'NCA': 'NI', 'NIG': 'NE', 'NGA': 'NG', 'NIR': 'NS', 'NOR': 'NO',
    'OMA': 'OM', 'PAK': 'PK', 'PLE': 'PS', 'PAN': 'PA', 'PNG': 'PG',
    'PAR': 'PY', 'PER': 'PE', 'PHI': 'PH', 'POL': 'PL', 'POR': 'PT',
    'PUR': 'PR', 'QAT': 'QA', 'IRL': 'IE', 'ROU': 'RO', 'RUS': 'RU',
    'RWA': 'RW', 'SAM': 'WS', 'SMR': 'SM', 'STP': 'ST', 'KSA': 'SA',
    'SCO': 'SQ', 'SEN': 'SN', 'SRB': 'RS', 'SEY': 'SC', 'SLE': 'SL',
    'SGP': 'SG', 'SVK': 'SK', 'SVN': 'SI', 'SOL': 'SB', 'SOM': 'SO',
    'RSA': 'ZA', 'SSD': 'SS', 'ESP': 'ES', 'SRI': 'LK', 'SKN': 'KN',
    'LCA': 'LC', 'VIN': 'VC', 'SDN': 'SD', 'SUR': 'SR', 'SWE': 'SE',
    'SUI': 'CH', 'SYR': 'SY', 'TAH': 'GP', 'TJK': 'TJ', 'TAN': 'TZ',
    'THA': 'TH', 'TLS': 'TL', 'TOG': 'TG', 'TGA': 'TO', 'TRI': 'TT',
    'TUN': 'TN', 'TUR': 'TR', 'TKM': 'TM', 'TCA': 'TC', 'VIR': 'VI',
    'USA': 'US', 'UGA': 'UG', 'UKR': 'UA', 'UAE': 'AE', 'URU': 'UY',
    'UZB': 'UZ', 'VAN': 'VU', 'VEN': 'VE', 'VIE': 'VN', 'WAL': 'WA',
    'YEM': 'YE', 'ZAM': 'ZM', 'ZIM': 'ZW'
}


fifa_rankings['country_code'] = fifa_rankings['country_code'].replace(fifa_to_elo)
elo_rankings = pd.merge(right=fifa_rankings, left=elo_features, on=['country_code', 'year'])


groups = elo_rankings.groupby('team_name')
elo_rankings['elo_change_1y'] = groups['elo'].diff()

data = pd.concat([long_df_2010, long_df_2014, long_df_2018, long_df_2022], axis=0)
data['year'] = data['date'].dt.year

data = pd.merge(left=data, right=elo_rankings, left_on=['year', 'team'], right_on=['year', 'team_name'], how='left')



In [35]:
home_df = data[['date', 'team', 'opponent', 'goals_for', 'goals_against', 'tournament',
       'result', 'match_played', 'average_goal', 'average_goal_conceded',
       'is_win', 'is_draw', 'win_rate', 'draw_rate', 'year', 'country_code',
       'elo', 'elo_change_1y', 'team_name', 'average_position',
       'average_points']]

away_df = data[['date', 'team', 'opponent', 'goals_for', 'goals_against', 'tournament',
       'result', 'match_played', 'average_goal', 'average_goal_conceded',
       'is_win', 'is_draw', 'win_rate', 'draw_rate', 'year', 'country_code',
       'elo', 'elo_change_1y', 'team_name', 'average_position',
       'average_points']]


df = pd.merge(home_df, away_df, left_on=['date', 'team', 'opponent'], 
              right_on=['date', 'opponent', 'team'], suffixes=('_home', '_away'))

df = df[df['team_home'] < df['team_away']].reset_index(drop=True)

#now let's perform a little feature engineering for having more informative data

df['elo_diff'] = df['elo_home'] - df['elo_away']
df['win_rate_diff'] = df['win_rate_home'] - df['win_rate_away']
df['avg_goal_diff'] = df['average_goal_home'] - df['average_goal_away']
df['avg_goal_conceded_diff'] = df['average_goal_conceded_home'] - df['average_goal_conceded_away']
df['position_diff'] = df['average_position_home'] - df['average_position_away']


In [36]:
df.to_csv('data_for_outcomes.csv')

## Training

With all the data collected, a model to predict the outcome of games had been trained on all the matches of non World Cups game. Later the same model was used to maximize the points that could be gained on the competition with respect to the challenge's rules.

In [37]:
data = pd.read_csv('data_for_outcomes.csv')
result_map = {'W': 0, 'D': 1, 'L': 2}
data['result'] = data['result_home'].map(result_map)


split_idx = int(len(data) * 0.75)
train = data.iloc[:split_idx]
test  = data.iloc[split_idx:]



features_col = ['average_goal_home', 'average_goal_conceded_home',
        'win_rate_home', 'draw_rate_home',
        'elo_home', 'elo_change_1y_home',
        'average_position_home', 'average_points_home',
       'average_goal_away', 'average_goal_conceded_away',  'win_rate_away', 'draw_rate_away',
        'elo_away', 'elo_change_1y_away', 
       'average_position_away', 'average_points_away', 'elo_diff',
       'win_rate_diff', 'avg_goal_diff', 'avg_goal_conceded_diff',
       'position_diff']

X_train = train[features_col]
X_test = test[features_col]

home_goals_train = train['goals_for_home'] 
home_goals_test = test['goals_for_home']

away_goals_train = train['goals_for_away']
away_goals_test = test['goals_for_away']


results_train = train['result']
results_test = test['result']


Now there are all the features and the 3 different targets: final result, home goals and away goals, and data split in train and test set.

### Outcome Model

A BayesSearch had been performed to find the best hyperparameters. Here the validation phase is present as a comment to speed up the code.

In [38]:
'''
grid = {'n_estimators': space.Integer(100, 1000),
        'max_depth': space.Integer(2, 8),          
        'min_child_weight': space.Integer(1, 10),   

        # Learning rate
        'learning_rate': space.Real(0.01, 0.3, prior='log-uniform'),

        'subsample': space.Real(0.5, 1.0),         
        'colsample_bytree': space.Real(0.5, 1.0),   

        # regulariztion
        'gamma': space.Real(0, 5),                  
        'reg_alpha': space.Real(1e-3, 10, prior='log-uniform'),   
        'reg_lambda': space.Real(1e-3, 10, prior='log-uniform')}

model = xgb.XGBClassifier(objective='multi:softprob', num_class=3, eval_metric='mlogloss', random_state=62, device='cuda')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=62)


search = BayesSearchCV(model, grid, cv=cv, scoring='f1_macro', random_state=62, n_iter=30, verbose=3)


sample_weights = compute_sample_weight(class_weight='balanced', y=results_train)

search.fit(X_train, results_train, sample_weight=sample_weights)
'''

result_model = xgb.XGBClassifier(objective='multi:softprob', num_class=3, 
                                 eval_metric='mlogloss', random_state=62, device='cuda',
                                 colsample_bytree=0.9890612524968385, gamma=4.424214745876272, 
                                 learning_rate=0.29999999999999993,
                                 max_depth=2, min_child_weight=10, n_estimators=821,
                                 reg_alpha=1.0743530404176647, reg_lambda=0.005529686917527576,
                                 subsample=0.6957731927171096)



sample_weights = compute_sample_weight(class_weight='balanced', y=results_train)

result_model.fit(X_train, results_train, sample_weight=sample_weights)


,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.9890612524968385
,device,'cuda'
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mlogloss'


## Exact result models

In [39]:
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline


#let's create a zero inflated model for home and away

## let's train a model that predicts if home team will score more than 0 goal
y_zero_home = (home_goals_train == 0).astype(int)

zero_pipeline_home = Pipeline([('imputer', SimpleImputer(strategy='median')), 
                               ('model', LogisticRegression(max_iter=1000))])

zero_pipeline_home.fit(X_train, y_zero_home)


## With the non zero example let's train a count model (Poisson)
non_zero_home = home_goals_train > 0

home_goals_model = xgb.XGBRegressor(objective='count:poisson', random_state=62, device='cuda', colsample_bytree=0.5006949013558107, 
                                    learning_rate=0.0742327727096257, max_depth=6, min_child_weight=5, n_estimators=646, reg_alpha=0.08934430748433142,
                                    reg_lambda=0.19731269080653094, subsample=0.7394555605641457, gamma=2.3273549099365622)

home_goals_model.fit(X_train[non_zero_home], home_goals_train[non_zero_home])


## Let's do the same as before with away teams
y_zero_away = (away_goals_train == 0).astype(int)

zero_pipeline_away = Pipeline([('imputer', SimpleImputer(strategy='median')), 
                               ('model', LogisticRegression(max_iter=1000))])

zero_pipeline_away.fit(X_train, y_zero_away)

#
non_zero_away = away_goals_train > 0

away_goals_model = xgb.XGBRegressor(objective='count:poisson', random_state=62, device='cuda', colsample_bytree=0.9211246085105704, 
                                    learning_rate=0.02620682159609111, max_depth=4, min_child_weight=8, n_estimators=558, reg_alpha=0.7719792817358803,
                                    reg_lambda=4.6466201813072345, subsample=0.5787751720855806, gamma=3.768499405863704)

away_goals_model.fit(X_train[non_zero_away], away_goals_train[non_zero_away])



## and now let's create a function that computes the expected goals as the probability of scoring more then 0 goal 
## multiplied by the number of predicted goals
def predict_count(X, zero, count):

    zero_probs = zero.predict_proba(X)[:, 1]
    goals = count.predict(X)
    
    expected_goals = (1 - zero_probs) * goals

    return expected_goals


home_goals_prediction = predict_count(X_test, zero_pipeline_home, home_goals_model)
away_goals_prediction = predict_count(X_test, zero_pipeline_away, away_goals_model)


c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\emanu\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://s

## Simulation

Starting from the given datasets, I will simulate the results of the group stage. Starting from that all the knock out matches will be simulated. The process of gathering all the data is the same seen for obtaining the training data.

In [40]:
results = pd.read_csv('data/results.csv')
results['date'] = pd.to_datetime(results['date'])
last_results_2026 = results[(results.date >= pd.to_datetime('2024-01-01')) &
(results.date <= pd.to_datetime('2026-06-10'))]

non_wc_2026 = last_results_2026[last_results_2026['tournament'] != 'FIFA World Cup']


home_2026 = non_wc_2026[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament']].copy()
home_2026.columns = ['date', 'team', 'opponent', 'goals_for', 'goals_against', 'tournament']

away_2026 = non_wc_2026[['date', 'away_team', 'home_team', 'away_score', 'home_score', 'tournament']].copy()
away_2026.columns = ['date', 'team', 'opponent', 'goals_for', 'goals_against', 'tournament']

long_df_2026 = pd.concat([home_2026, away_2026], axis=0)
long_df_2026 = long_df_2026.sort_values(by=['team', 'date']).reset_index(drop=True)

groups = long_df_2026.groupby('team')

mask = [long_df_2026.goals_for > long_df_2026.goals_against,
        long_df_2026.goals_for == long_df_2026.goals_against,
        long_df_2026.goals_for < long_df_2026.goals_against]

long_df_2026['result'] = np.select(mask, ['W', 'D', 'L'], default='N/A')
long_df_2026['match_played'] = groups.cumcount()
long_df_2026['average_goal'] = groups['goals_for'].transform(lambda x: x.cumsum().shift(1)) / long_df_2026['match_played'].fillna(0)
long_df_2026['average_goal_conceded'] = groups['goals_against'].transform(lambda x: x.cumsum().shift(1)) / long_df_2026['match_played'].fillna(0)
long_df_2026['is_win'] = np.where(long_df_2026['result'] == 'W', 1, 0)
long_df_2026['is_draw'] = np.where(long_df_2026['result'] == 'D', 1, 0)
long_df_2026['win_rate'] = (groups['is_win'].transform(lambda x: x.cumsum().shift(1)) / long_df_2026['match_played']).fillna(0)
long_df_2026['draw_rate'] = (groups['is_draw'].transform(lambda x: x.cumsum().shift(1)) / long_df_2026['match_played']).fillna(0)

final_features_2026 = long_df_2026.sort_values('date').groupby('team').last().reset_index()


In [41]:
id_to_date_2026_fixed = {"id14870": "2025-09-18"}

def get_fifa_ranking(date_id, date):
    url = f"https://inside.fifa.com/api/ranking-overview?lang=en&dateId={date_id}"
    response = requests.get(url, headers={
        'User-Agent': 'Mozilla/5.0',
        'Accept': 'application/json',
        'Referer': 'https://inside.fifa.com/fifa-world-ranking/men'
    })
    data = response.json()
    records = []
    for item in data['rankings']:
        r = item['rankingItem']
        records.append({
            'date':         pd.to_datetime(date),
            'country_code': r['countryCode'],
            'team_name':    r['name'],
            'fifa_rank':    r['rank'],
            'fifa_points':  r['totalPoints'],
        })
    return pd.DataFrame(records)

all_dfs_2026 = []
for date_id, date in id_to_date_2026_fixed.items():
    df = get_fifa_ranking(date_id, date)
    all_dfs_2026.append(df)
    

fifa_rankings_2026 = pd.concat(all_dfs_2026, ignore_index=True)
fifa_rankings_2026 = fifa_rankings_2026.sort_values(['team_name', 'date']).reset_index(drop=True)

latest_ranking_2026 = fifa_rankings_2026.sort_values('date').groupby('team_name').last().reset_index()


In [42]:
latest_ranking_2026['year'] = latest_ranking_2026['date'].dt.year

groups = latest_ranking_2026.groupby(['team_name', 'year'])

latest_ranking_2026['average_position'] = groups['fifa_rank'].transform('mean')
latest_ranking_2026['average_points'] = groups['fifa_points'].transform('mean')

fifa_rankings_2026 = latest_ranking_2026[['year', 'country_code', 'team_name', 'average_position', 'average_points']].copy()

fifa_rankings_2026 = fifa_rankings_2026.drop_duplicates(subset=['year', 'team_name'], keep='last')


In [43]:
url = "https://www.eloratings.net/2026.tsv"
response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
response.encoding = 'utf-8'
df_elo_2026 = pd.read_csv(StringIO(response.text), sep='\t', header=None)
df_elo_2026.columns = columns  
df_elo_2026['year'] = 2026

elo_features_2026 = df_elo_2026[['country_code', 'elo', 'elo_change_1y', 'year']]

In [44]:

fifa_to_elo = {
    'AFG': 'AF', 'ALB': 'AL', 'ALG': 'DZ', 'ASA': 'AS', 'AND': 'AD',
    'ANG': 'AO', 'AIA': 'AI', 'ATG': 'AG', 'ARG': 'AR', 'ARM': 'AM',
    'ARU': 'AW', 'AUS': 'AU', 'AUT': 'AT', 'AZE': 'AZ', 'BAH': 'BS',
    'BHR': 'BH', 'BAN': 'BD', 'BRB': 'BB', 'BLR': 'BY', 'BEL': 'BE',
    'BLZ': 'BZ', 'BEN': 'BJ', 'BER': 'BM', 'BHU': 'BT', 'BOL': 'BO',
    'BIH': 'BA', 'BOT': 'BW', 'BRA': 'BR', 'VGB': 'VG', 'BRU': 'BN',
    'BUL': 'BG', 'BFA': 'BF', 'BDI': 'BI', 'CPV': 'CV', 'CAM': 'KH',
    'CMR': 'CM', 'CAN': 'CA', 'CAY': 'KY', 'CTA': 'CF', 'CHA': 'TD',
    'CHI': 'CL', 'CHN': 'CN', 'TPE': 'TW', 'COL': 'CO', 'COM': 'KM',
    'CGO': 'CG', 'COD': 'CD', 'COK': 'CK', 'CRC': 'CR', 'CRO': 'HR',
    'CUB': 'CU', 'CUW': 'CW', 'CYP': 'CY', 'CZE': 'CZ', 'CIV': 'CI',
    'DEN': 'DK', 'DJI': 'DJ', 'DMA': 'DM', 'DOM': 'DO', 'ECU': 'EC',
    'EGY': 'EG', 'SLV': 'SV', 'ENG': 'EN', 'EQG': 'GQ', 'ERI': 'ER',
    'EST': 'EE', 'SWZ': 'SZ', 'ETH': 'ET', 'MKD': 'MK', 'FRO': 'FO',
    'FIJ': 'FJ', 'FIN': 'FI', 'FRA': 'FR', 'GAB': 'GA', 'GAM': 'GM',
    'GEO': 'GE', 'GER': 'DE', 'GHA': 'GH', 'GIB': 'GI', 'GRE': 'GR',
    'GRN': 'GD', 'GUM': 'GU', 'GUA': 'GT', 'GUI': 'GN', 'GNB': 'GW',
    'GUY': 'GY', 'HAI': 'HT', 'HON': 'HN', 'HKG': 'HK', 'HUN': 'HU',
    'IRN': 'IR', 'ISL': 'IS', 'IND': 'IN', 'IDN': 'ID', 'IRQ': 'IQ',
    'ISR': 'IL', 'ITA': 'IT', 'JAM': 'JM', 'JPN': 'JP', 'JOR': 'JO',
    'KAZ': 'KZ', 'KEN': 'KE', 'PRK': 'KP', 'KOR': 'KR', 'KOS': 'KO',
    'KUW': 'KW', 'KGZ': 'KG', 'LAO': 'LA', 'LVA': 'LV', 'LBN': 'LB',
    'LIB': 'LY', 'LES': 'LS', 'LBR': 'LR', 'LBY': 'LY', 'LIE': 'LI',
    'LTU': 'LT', 'LUX': 'LU', 'MAC': 'MO', 'MAD': 'MG', 'MWI': 'MW',
    'MAS': 'MY', 'MDV': 'MV', 'MLI': 'ML', 'MLT': 'MT', 'MTN': 'MR',
    'MRI': 'MU', 'MEX': 'MX', 'MDA': 'MD', 'MNG': 'MN', 'MNE': 'ME',
    'MSR': 'MS', 'MAR': 'MA', 'MOZ': 'MZ', 'MYA': 'MM', 'NAM': 'NM',
    'NEP': 'NP', 'NED': 'NL', 'ANT': 'AN', 'NCL': 'NC', 'NZL': 'NZ',
    'NCA': 'NI', 'NIG': 'NE', 'NGA': 'NG', 'NIR': 'NS', 'NOR': 'NO',
    'OMA': 'OM', 'PAK': 'PK', 'PLE': 'PS', 'PAN': 'PA', 'PNG': 'PG',
    'PAR': 'PY', 'PER': 'PE', 'PHI': 'PH', 'POL': 'PL', 'POR': 'PT',
    'PUR': 'PR', 'QAT': 'QA', 'IRL': 'IE', 'ROU': 'RO', 'RUS': 'RU',
    'RWA': 'RW', 'SAM': 'WS', 'SMR': 'SM', 'STP': 'ST', 'KSA': 'SA',
    'SCO': 'SQ', 'SEN': 'SN', 'SRB': 'RS', 'SEY': 'SC', 'SLE': 'SL',
    'SGP': 'SG', 'SVK': 'SK', 'SVN': 'SI', 'SOL': 'SB', 'SOM': 'SO',
    'RSA': 'ZA', 'SSD': 'SS', 'ESP': 'ES', 'SRI': 'LK', 'SKN': 'KN',
    'LCA': 'LC', 'VIN': 'VC', 'SDN': 'SD', 'SUR': 'SR', 'SWE': 'SE',
    'SUI': 'CH', 'SYR': 'SY', 'TAH': 'GP', 'TJK': 'TJ', 'TAN': 'TZ',
    'THA': 'TH', 'TLS': 'TL', 'TOG': 'TG', 'TGA': 'TO', 'TRI': 'TT',
    'TUN': 'TN', 'TUR': 'TR', 'TKM': 'TM', 'TCA': 'TC', 'VIR': 'VI',
    'USA': 'US', 'UGA': 'UG', 'UKR': 'UA', 'UAE': 'AE', 'URU': 'UY',
    'UZB': 'UZ', 'VAN': 'VU', 'VEN': 'VE', 'VIE': 'VN', 'WAL': 'WA',
    'YEM': 'YE', 'ZAM': 'ZM', 'ZIM': 'ZW'
}

name_corrections = {
    'Czech Republic': 'Czechia',
    'DR Congo': 'Congo DR',
    'Iran': 'IR Iran',
    'Ivory Coast': "Côte d'Ivoire",
    'South Korea': 'Korea Republic',
    'Turkey': 'Türkiye',
    'United States': 'USA',
    'North Macedonia': 'North Macedonia',  
    'Cape Verde': 'Cabo Verde',
    'Kyrgyzstan': 'Kyrgyz Republic',
    'Gambia': 'The Gambia',
    'Saint Kitts and Nevis': 'St. Kitts and Nevis',
    'Saint Lucia': 'St. Lucia',
    'Saint Vincent and the Grenadines': 'St. Vincent and the Grenadines',
    'United States Virgin Islands': 'US Virgin Islands',
    'Martinique': 'Martinique',   
    'Sint Maarten': 'Sint Maarten', 
    'Saint Martin': 'Saint Martin',  
    'Bonaire': 'Bonaire',  
}



fifa_rankings_2026['year'] = 2026
final_features_2026['team_corrected'] = final_features_2026['team'].replace(name_corrections)
fifa_rankings_2026['country_code'] = fifa_rankings_2026['country_code'].replace(fifa_to_elo)
elo_rankings_2026 = pd.merge(right=fifa_rankings_2026, left=elo_features_2026, on=['country_code', 'year'])

data_2026 = pd.merge(left=final_features_2026, right=elo_rankings_2026,
                     left_on='team_corrected', right_on='team_name',how='left')

data_2026['elo_change_1y'] = data_2026['elo_change_1y'].astype(str)\
    .str.replace('−', '-', regex=False)\
    .str.replace('+', '', regex=False)\
    .str.strip()
data_2026['elo_change_1y'] = pd.to_numeric(data_2026['elo_change_1y'], errors='coerce')  

kaggle_to_playoff = {
    'Bosnia and Herzegovina': 'Bosnia and Herzegovina',  
    'Sweden': 'Sweden',                                   
    'Türkiye': 'Turkey',                                  
    'Czechia': 'Czech Republic',                          
    'DR Congo': 'DR Congo',                               
    'Curaçao': 'Curaçao',                                 
}

extra_rows = []
for playoff_name, kaggle_name in kaggle_to_playoff.items():
    if playoff_name != kaggle_name:  
        row = data_2026[data_2026['team'] == kaggle_name]
        if len(row) > 0:
            row = row.copy()
            row['team'] = playoff_name
            extra_rows.append(row)

if extra_rows:
    data_2026 = pd.concat([data_2026] + extra_rows, ignore_index=True)

### Group stages

Now it's time for the actual simulation. At first the group stages will be simulated after having added the last nations to the dataset.

The get_match_features function retrieves the pre-computed features for a given home and away team from the 2026 feature dataset and assembles them into a single-row DataFrame ready for model inference. For each match these features are used to simulate the final result and the winner.

The results are then used to build a standings table for each group, sorting teams by total points, goal difference, and goals scored. The top two teams from each group advance to the knockout stage, along with the eight best third-placed teams.

In [45]:
group_fixtures = pd.read_csv('data/group_fixtures.csv')


playoff_map = {'UEFA Playoff A': 'Bosnia and Herzegovina','UEFA Playoff B': 'Sweden',
               'UEFA Playoff C': 'Türkiye', 'UEFA Playoff D': 'Czechia', 
               'FIFA Playoff 1': 'DR Congo', 'FIFA Playoff 2': 'Curaçao'}

group_fixtures['home_team'] = group_fixtures['home_team'].replace(playoff_map)
group_fixtures['away_team'] = group_fixtures['away_team'].replace(playoff_map)

datacamp_to_kaggle = {
    'USA': 'United States',
    "Côte d'Ivoire": 'Ivory Coast',
    'Türkiye': 'Turkey',
    'Czechia': 'Czech Republic',
    'Cabo Verde': 'Cape Verde'
}

def get_match_features(home_team, away_team, team_features):
    feat_cols = ['average_goal', 'average_goal_conceded', 'win_rate',
                 'draw_rate', 'elo', 'elo_change_1y',
                 'average_position', 'average_points']
    
    # Ricerca flessibile: controlla sia il nome originale Kaggle che quello corretto (DataCamp/FIFA)
    home = team_features[(team_features['team'] == home_team) | 
                        (team_features['team_corrected'] == home_team)]
    away = team_features[(team_features['team'] == away_team) |
                        (team_features['team_corrected'] == away_team)]
    
    if len(home) == 0 or len(away) == 0:
        return None
    
    row = {}
    for col in feat_cols:
        row[f'{col}_home'] = float(home[col].values[0]) 
        row[f'{col}_away'] = float(away[col].values[0])
    
    row['elo_diff'] = row['elo_home'] - row['elo_away']
    row['win_rate_diff'] = row['win_rate_home'] - row['win_rate_away']
    row['avg_goal_diff'] = row['average_goal_home'] - row['average_goal_away']
    row['avg_goal_conceded_diff'] = row['average_goal_conceded_home'] - row['average_goal_conceded_away']
    row['position_diff'] = row['average_position_home'] - row['average_position_away']
    
    return pd.DataFrame([row])

In [46]:
def simulate_group_stage(fixtures, features):
    base_features = ['average_goal', 'average_goal_conceded', 'win_rate', 
                     'draw_rate', 'elo', 'elo_change_1y', 
                     'average_position', 'average_points']
    
    features_clean = features.drop_duplicates(subset=['team']).dropna(subset=base_features)
    
    predictions = fixtures.copy()
    home_goals_list, away_goals_list, winners_list = [], [], []
    result_map_inv = {0: 'home', 1: 'draw', 2: 'away'}

    for _, match in fixtures.iterrows():
        X = get_match_features(match['home_team'], match['away_team'], features_clean)
        
        if X is None or X.empty:
            home_goals_list.append(1); away_goals_list.append(1); winners_list.append('draw')
            continue
            
        X = X[features_col]
        outcome = result_model.predict(X)[0]
        home_g = round(predict_count(X, zero_pipeline_home, home_goals_model)[0])
        away_g = round(predict_count(X, zero_pipeline_away, away_goals_model)[0])
        
        winner = result_map_inv[outcome]
        
        if winner == 'home' and home_g <= away_g:
            home_g = away_g + 1
        elif winner == 'away' and away_g <= home_g:
            away_g = home_g + 1
        elif winner == 'draw' and home_g != away_g:
            home_g = away_g = (home_g + away_g) // 2
            
        home_goals_list.append(home_g)
        away_goals_list.append(away_g)
        winners_list.append(winner)

    predictions['predicted_home_goals'] = home_goals_list
    predictions['predicted_away_goals'] = away_goals_list
    predictions['winning_team'] = winners_list
    predictions['corners'] = 10
    predictions['yellow_cards'] = 3
    predictions['red_cards'] = 0
    
    return predictions

    
group_predictions = simulate_group_stage(group_fixtures, data_2026)

In [47]:
group_predictions.to_csv('data/group_predictions.csv')

In [48]:
group_results = pd.read_csv('data/group_predictions.csv')

home_results = group_results[['group', 'home_team', 'predicted_home_goals', 'predicted_away_goals', 'winning_team']].copy()
home_results.columns = ['group', 'team', 'goals_for', 'goals_against', 'result']
home_results['points'] = home_results['result'].map({'home': 3, 'draw': 1, 'away': 0})

away_results = group_results[['group', 'away_team', 'predicted_away_goals', 'predicted_home_goals', 'winning_team']].copy()
away_results.columns = ['group', 'team', 'goals_for', 'goals_against', 'result']
away_results['points'] = away_results['result'].map({'away': 3, 'draw': 1, 'home': 0})

all_results = pd.concat([home_results, away_results], axis=0)
all_results['goal_diff'] = all_results['goals_for'] - all_results['goals_against']

standings = all_results.groupby(['group', 'team']).agg(total_points=('points', 'sum'),
                                                       total_goal_diff=('goal_diff', 'sum'),
                                                       total_goals=('goals_for', 'sum')).reset_index()

standings = standings.sort_values(['group', 'total_points', 'total_goal_diff', 'total_goals'],
                                  ascending=[True, False, False, False])



Let's keep the two best team of every group.

In [49]:
qualifiers = {}
for group, group_df in standings.groupby('group'):
    teams = group_df['team'].tolist()
    qualifiers[f'Winner Group {group}'] = teams[0]
    qualifiers[f'Runner-up Group {group}'] = teams[1]



### Knock-outs

The knockout stage follows the official bracket.
For each knockout match, the slot names are dynamically resolved based on the results of previous rounds. The simulate_match function predicts the outcome using the same models as before. If the outcome model predicts a draw the winner is determined by comparing the predicted probabilities of a home win and an away win, simulating a penalty shootout scenario.

In [50]:
knockout_slots = pd.read_csv('data/knockout_slots.csv')

# selecting all the thirds
third_place = {}
for group, group_df in standings.groupby('group'):
    teams = group_df['team'].tolist()
    if len(teams) >= 3:
        third_place[f'3rd Group {group}'] = {
            'team': teams[2],
            'points': group_df.iloc[2]['total_points'],
            'goal_diff': group_df.iloc[2]['total_goal_diff'],
            'goals': group_df.iloc[2]['total_goals']
        }

# getting 8 best thirds
thirds_df = pd.DataFrame(third_place).T.reset_index()
thirds_df.columns = ['slot', 'team', 'points', 'goal_diff', 'goals']
thirds_df = thirds_df.sort_values(['points', 'goal_diff', 'goals'], ascending=[False, False, False]).head(8)

best_thirds = thirds_df['team'].tolist()

best_third_matches = knockout_slots[knockout_slots['slot_home'].str.contains('Best 3rd') | 
                                    knockout_slots['slot_away'].str.contains('Best 3rd')]['match_id'].tolist()

best_third_idx = 0
for _, row in knockout_slots.iterrows():
    if 'Best 3rd' in str(row['slot_home']):
        qualifiers[f"Best3rd Match {row['match_id']}"] = best_thirds[best_third_idx]
        knockout_slots.loc[knockout_slots['match_id'] == row['match_id'], 'slot_home'] = f"Best3rd Match {row['match_id']}"
        best_third_idx += 1
    if 'Best 3rd' in str(row['slot_away']):
        qualifiers[f"Best3rd Match {row['match_id']}"] = best_thirds[best_third_idx]
        knockout_slots.loc[knockout_slots['match_id'] == row['match_id'], 'slot_away'] = f"Best3rd Match {row['match_id']}"
        best_third_idx += 1

# function for simulating knockout matches with logical handling of draws and possible incoherence 
# between winner and score
def simulate_knockout_match(home_team, away_team, team_features):
    X = get_match_features(home_team, away_team, team_features)
    
    if X is None or X.empty:
        return 1, 0, 'home', False
        
    X = X[features_col]
    probs = result_model.predict_proba(X)[0]  
    outcome = result_model.predict(X)[0]
    
    home_g = round(predict_count(X, zero_pipeline_home, home_goals_model)[0])
    away_g = round(predict_count(X, zero_pipeline_away, away_goals_model)[0])
    
    if outcome == 1: 
        penalties = True
        home_g = away_g = (home_g + away_g) // 2  
        match_winner = 'home' if probs[0] > probs[2] else 'away'
        
    elif outcome == 0:  
        penalties = False
        match_winner = 'home'
        if home_g <= away_g:
            home_g = away_g + 1
            
    else:  
        penalties = False
        match_winner = 'away'
        if away_g <= home_g:
            away_g = home_g + 1
            
    return home_g, away_g, match_winner, penalties

knockout_results = []
match_winners = {}  

for _, row in knockout_slots.iterrows():
    match_id = row['match_id']
    slot_home = row['slot_home']
    slot_away = row['slot_away']
    
    home_team = qualifiers.get(slot_home) or match_winners.get(slot_home)
    away_team = qualifiers.get(slot_away) or match_winners.get(slot_away)
    
    if home_team is None or away_team is None:
        knockout_results.append({
            'match_id': match_id, 'round': row['round'],
            'predicted_home_team': home_team or 'TBD', 'predicted_away_team': away_team or 'TBD',
            'predicted_home_goals': 1, 'predicted_away_goals': 0,
            'corners': 10, 'yellow_cards': 3, 'red_cards': 0,
            'match_winner': 'home', 'penalties': False
        })
        continue
    
    home_g, away_g, match_winner, penalties = simulate_knockout_match(home_team, away_team, data_2026)
    
    if match_winner == 'home':
        winner_team = home_team
        loser_team = away_team
    else:
        winner_team = away_team
        loser_team = home_team
        
    match_winners[f'Winner Match {match_id}'] = winner_team
    match_winners[f'Loser Match {match_id}'] = loser_team
    
    knockout_results.append({
        'match_id': match_id,
        'round': row['round'],
        'predicted_home_team': home_team,
        'predicted_away_team': away_team,
        'predicted_home_goals': home_g,
        'predicted_away_goals': away_g,
        'corners': 10,
        'yellow_cards': 3,
        'red_cards': 0,
        'match_winner': match_winner,
        'penalties': penalties
    })

knockout_predictions = pd.DataFrame(knockout_results)

In [51]:
knockout_predictions.to_csv('data/knockout_predictions.csv')

All the predictions are saved in the group_predictions and knockout_predictions dataframes. 

In [52]:
group_predictions

,match_id,group,home_team,away_team,date_utc,venue,predicted_home_goals,predicted_away_goals,winning_team,corners,yellow_cards,red_cards
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City",2,1,home,10,3,0
1,2,A,South Korea,Czechia,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara",2,1,home,10,3,0
2,3,B,Canada,Bosnia and Herzegovina,2026-06-12T19:00:00Z,"BMO Field, Toronto",2,1,home,10,3,0
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles",1,1,draw,10,3,0
4,5,D,Australia,Türkiye,2026-06-13T04:00:00Z,"BC Place, Vancouver",1,2,away,10,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...
67,68,L,Croatia,Ghana,2026-06-27T21:00:00Z,"Lincoln Financial Field, Philadelphia",2,1,home,10,3,0
68,69,K,Colombia,Portugal,2026-06-27T23:30:00Z,"Hard Rock Stadium, Miami",2,1,home,10,3,0
69,70,K,DR Congo,Uzbekistan,2026-06-27T23:30:00Z,"Mercedes-Benz Stadium, Atlanta",1,1,draw,10,3,0
70,71,J,Algeria,Austria,2026-06-28T02:00:00Z,"GEHA Field at Arrowhead Stadium, Kansas City",2,1,home,10,3,0


In [53]:
standings

,group,team,total_points,total_goal_diff,total_goals
1,A,Mexico,9,3,6
3,A,South Korea,6,1,5
0,A,Czechia,3,-1,4
2,A,South Africa,0,-3,3
7,B,Switzerland,9,4,6
5,B,Canada,6,1,5
4,B,Bosnia and Herzegovina,3,-1,4
6,B,Qatar,0,-4,2
8,C,Brazil,5,1,4
10,C,Morocco,5,1,4


In [54]:
best_thirds

['Scotland',
 "Côte d'Ivoire",
 'Iran',
 'Czechia',
 'Bosnia and Herzegovina',
 'Senegal',
 'Austria',
 'Panama']

In [55]:
knockout_predictions

,match_id,round,predicted_home_team,predicted_away_team,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,match_winner,penalties
0,73,Round of 32,South Korea,Canada,1,1,10,3,0,home,True
1,74,Round of 32,Brazil,Netherlands,1,1,10,3,0,home,True
2,75,Round of 32,Germany,Scotland,2,1,10,3,0,home,False
3,76,Round of 32,Japan,Morocco,2,1,10,3,0,home,False
4,77,Round of 32,Ecuador,Norway,1,1,10,3,0,home,True
5,78,Round of 32,France,Côte d'Ivoire,2,1,10,3,0,home,False
6,79,Round of 32,Mexico,Iran,2,1,10,3,0,home,False
7,80,Round of 32,England,Czechia,2,1,10,3,0,home,False
8,81,Round of 32,Belgium,Bosnia and Herzegovina,2,1,10,3,0,home,False
9,82,Round of 32,Türkiye,Senegal,2,1,10,3,0,home,False
